# D3 Rana — GraphRAG Executor: Subgraph -> Chunks -> Answer -> Citations

**Owner:** Rana  
**Status:** Implemented (pipeline code complete; cells below are unexecuted pending a live Neo4j + Gemini run — see Setup).

**Main evidence:** the actual 4-step GraphRAG execution path with visible graph paths and citations, for 3 query patterns that work well with the knowledge graph plus 3 documented failure cases.

This notebook is the canonical generator of `reports/tables/d3_graph_guided_results.csv`. The shared notebook `D3_graphrag_eval_safety.ipynb` ("Rana block") shows one condensed example from this notebook plus the cross-team write-up.

In [1]:
# Setup — bootstrap project root, imports of the real pipeline (no more TODOs below).
import os, sys, warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() in {"notebooks", "notebook"}:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)  # required because config paths are repo-relative

REPORTS_TABLES = PROJECT_ROOT / "reports" / "tables"
REPORTS_TABLES.mkdir(parents=True, exist_ok=True)

from src.rag.graphrag_executor import (
    GraphRAGExecutor, SubgraphSelector, GraphExpander,
    HybridBlender, AnswerGenerator, BlendedChunk, GraphHit,
    load_local_env, redact_sensitive_text,
)
from src.rag.prompt_builder import PromptBuilder
from src.rag.citation_builder import CitationBuilder
from src.graph.cypher_queries import GRAPHRAG_REASONING

print("Project root:", PROJECT_ROOT)
print("Reports/tables:", REPORTS_TABLES)
print("Imports OK")


Project root: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent
Reports/tables: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables
Imports OK


### Optional emergency reset

Run the next cell only when the notebook kernel has stale environment variables or an old Neo4j URI. It rebuilds the executor and keeps Gemini disabled for stable D3 evidence generation.


In [2]:
# Emergency reset cell — self-contained.
# Run this if the notebook shows the old Neo4j URI or if earlier setup cells were skipped.
import os
import sys
import importlib
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() in {"notebooks", "notebook"}:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import src.rag.graphrag_executor as ge
ge = importlib.reload(ge)  # reloads the patched source file, not stale kernel code
GraphRAGExecutor = ge.GraphRAGExecutor
load_local_env = ge.load_local_env

# Force-reload local .env over any stale kernel values.
load_local_env(PROJECT_ROOT, override=True)
if os.environ.get("NEO4J_USERNAME"):
    os.environ["NEO4J_USER"] = os.environ["NEO4J_USERNAME"]

print("Using Neo4j URI scheme:", os.environ.get("NEO4J_URI", "").split("://", 1)[0])
print("NEO4J_USER set:", bool(os.environ.get("NEO4J_USER")))
print("NEO4J_DATABASE:", os.environ.get("NEO4J_DATABASE", "MISSING"))

executor = GraphRAGExecutor.from_config(str(PROJECT_ROOT / "configs" / "config.yaml"))

# Disable Gemini routing and answer generation for stable Rana D3 execution.
executor.selector.llm_classify = False
executor.selector.gemini_api_key = ""
executor.selector.classify_intent = lambda query: executor.selector._classify_with_keywords(query)
executor.generator.gemini_api_key = ""

def _mock_gemini_answer(prompt: str) -> str:
    return "[MOCK - Gemini disabled for stable Rana D3 run]\n\n" + prompt[:1200]

executor.generator._call_gemini = _mock_gemini_answer

selector_db = getattr(executor.selector, "database", None)
with (executor.selector.driver.session(database=selector_db) if selector_db else executor.selector.driver.session()) as session:
    print("Node count:", session.run("MATCH (n) RETURN count(n) AS c").single()["c"])
    print("Finding count:", session.run("MATCH (f:Finding) RETURN count(f) AS c").single()["c"])

print("Executor reset complete. If this prints 552 nodes / 15 findings, run the example cells again.")


Using Neo4j URI scheme: neo4j+ssc
NEO4J_USER set: True
NEO4J_DATABASE: cb62b391


Node count: 552
Finding count: 15
Executor reset complete. If this prints 552 nodes / 15 findings, run the example cells again.


## 1. Before coding: what does the graph add beyond vector search, and when is fallback safer?

**What the graph adds that vector search cannot:**
- **Structured multi-hop paths.** A query like *"What renewable targets has the UAE committed to?"* needs `Country ? Policy ? Target`. Vector search returns chunks that *mention* UAE and renewables, but it has no notion of which Policy a Target belongs to — that relationship only exists as a graph edge, not as text proximity.
- **Confidence-annotated, page-anchored evidence.** `MITIGATES`, `IMPACTS`, and `OCCURS_IN` edges carry a `confidence` and `evidence_page` property set during ingestion. This gives a citation a page number *before* any chunk similarity search happens.
- **Precision on named-entity questions.** Country / Policy / Technology / Risk questions resolve to exact nodes via `ENTITY_ALIASES`, so the answer is grounded in the *correct* entity rather than the most similar-sounding text.

**When hybrid fallback is safer than graph retrieval:**
1. **No entity extracted** — if the LLM classifier (or keyword router) can't resolve any `country_id` / `risk_id` / `tech_id` / `sector_id`, there is nothing to anchor a Cypher template to. Forcing a query anyway just returns 0 rows.
2. **Entity not in the alias table or not in the corpus graph** — e.g. "Pacific Islands" has no `Country`/`Region` node, so any Cypher template parameterised on it returns empty.
3. **Graph expansion is too broad** — e.g. a risk/sector-free `Finding` query with no filters returns 50+ rows that collapse into near-duplicate chunks from the same 2–3 source documents. This is not wrong, just redundant — hybrid's semantic diversity covers the topic better.
4. **Query is methodological, not entity-based** — e.g. "gradient boosting for wind forecasting" has no Technology/Risk node to traverse; the schema models climate entities, not ML methods.
5. **Cypher times out** (`cypher_timeout_sec`, default 10s in `configs/config.yaml`) — silent fallback rather than blocking the answer.

`GraphRAGExecutor` implements this as two **parallel lanes** (graph + hybrid), not a strict waterfall: if Stage A/B return nothing, Stage C still has hybrid evidence to blend from, and `retrieval_type` is set to `hybrid_fallback`.

## 2. Requirement mapping

Brief requirement covered by this notebook:

1. Choose subgraph by Cypher — `SubgraphSelector` (Stage A).
2. Expand to supporting chunks — `GraphExpander` (Stage B).
3. Hybrid blend and rerank — `HybridBlender` (Stage C).
4. Answer with citations and page ranges — `PromptBuilder` + `AnswerGenerator` + `CitationBuilder` (Stage D).

Files implemented/used:

- `src/rag/graphrag_executor.py` — all 4 stages, orchestrated by `GraphRAGExecutor`.
- `src/rag/prompt_builder.py` — two-section grounded prompt (graph evidence first, hybrid supplementary).
- `src/rag/citation_builder.py` — resolves chunks/Finding nodes to page-level APA citations; also produces the CSV row schema (incl. `fallback_used`, `failure_note`, `latency_sec`).
- `src/graph/cypher_queries.py` — `GRAPHRAG_REASONING`, the 5 parameterised Cypher templates this notebook exercises.
- `reports/tables/d3_graph_guided_results.csv` — output of this notebook.
- `reports/member3_d3_graphrag_executor_section.md` — write-up.

**Design decision (waterfall vs. parallel lanes):** the graph path (Stages A–B) and the hybrid path (BM25 + dense) run independently and merge at Stage C, so a Stage A failure (no Cypher hits) never blanks the answer — it degrades to `hybrid_fallback`. Full alternatives analysis in `D3_DESIGN_DOCUMENT.md`.

In [3]:
# Initialise the pipeline from .env — no secrets are written in this notebook.
# This cell force-loads .env with override=True so stale kernel values cannot reuse
# an old Neo4j Aura URI.
USE_GEMINI_ANSWERS = False
DISABLE_LLM_INTENT_CLASSIFIER = True

load_local_env(PROJECT_ROOT, override=True)
if os.environ.get("NEO4J_USERNAME"):
    os.environ["NEO4J_USER"] = os.environ["NEO4J_USERNAME"]

required = ["NEO4J_URI", "NEO4J_USER", "NEO4J_PASSWORD"]
missing = [k for k in required if not os.environ.get(k)]
if missing:
    raise RuntimeError(
        "Missing Neo4j settings in project .env: " + ", ".join(missing) +
        ". Put the Aura credentials in the ignored .env file, not in the notebook."
    )

print("Credential status after forced .env reload:")
for key in ["NEO4J_URI", "NEO4J_USER", "NEO4J_DATABASE", "NEO4J_PASSWORD", "GEMINI_API_KEY"]:
    value = os.environ.get(key)
    if key.endswith("PASSWORD") or key.endswith("KEY"):
        print(f"- {key}:", "SET" if value else "MISSING")
    elif key == "NEO4J_URI" and value:
        print(f"- {key}:", value.split("://", 1)[0] + "://[redacted]")
    else:
        print(f"- {key}:", value if value else "MISSING")

executor = GraphRAGExecutor.from_config(str(PROJECT_ROOT / "configs" / "config.yaml"))

# Disable Gemini routing/classification.
if DISABLE_LLM_INTENT_CLASSIFIER and hasattr(executor, "selector"):
    executor.selector.llm_classify = False
    executor.selector.gemini_api_key = ""
    executor.selector.classify_intent = lambda query: executor.selector._classify_with_keywords(query)
    print("GraphRAG intent classifier disabled; routing uses deterministic keyword logic.")

# Disable Gemini answer generation for stable submission reruns.
if not USE_GEMINI_ANSWERS and hasattr(executor, "generator"):
    executor.generator.gemini_api_key = ""

    def _mock_gemini_answer(prompt: str) -> str:
        return "[MOCK - Gemini disabled for stable Rana D3 run]\n\n" + prompt[:1200]

    executor.generator._call_gemini = _mock_gemini_answer
    print("Gemini answer generation disabled for stable run; answers show the grounded mock prompt.")

citation_builder = CitationBuilder(
    chunk_store_path=str(PROJECT_ROOT / "data" / "sample" / "sample_chunks.json"),
    metadata_csv_path=str(PROJECT_ROOT / "data" / "metadata" / "papers_metadata.csv"),
)

selector_db = getattr(executor.selector, "database", None)
with (executor.selector.driver.session(database=selector_db) if selector_db else executor.selector.driver.session()) as session:
    GRAPH_NODE_COUNT = session.run("MATCH (n) RETURN count(n) AS c").single()["c"]
    GRAPH_FINDING_COUNT = session.run("MATCH (f:Finding) RETURN count(f) AS c").single()["c"]

print("GraphRAGExecutor ready.")
print("Neo4j graph node count:", GRAPH_NODE_COUNT)
print("Neo4j Finding count:", GRAPH_FINDING_COUNT)
print("Answer mode:", "gemini" if USE_GEMINI_ANSWERS else "mock_prompt_no_gemini")


Credential status after forced .env reload:
- NEO4J_URI: neo4j+ssc://[redacted]
- NEO4J_USER: cb62b391
- NEO4J_DATABASE: cb62b391
- NEO4J_PASSWORD: SET
- GEMINI_API_KEY: SET


GraphRAG intent classifier disabled; routing uses deterministic keyword logic.
Gemini answer generation disabled for stable run; answers show the grounded mock prompt.


GraphRAGExecutor ready.
Neo4j graph node count: 552
Neo4j Finding count: 15
Answer mode: mock_prompt_no_gemini


---
### Helper: `display_result()`
Renders the full execution trace for one query: query → Cypher → graph path → supporting chunks → blended context → answer → citations.

In [4]:
def display_result(result, label=""):
    """Render the full GraphRAG execution trace for a GraphRAGResult."""
    from IPython.display import display, Markdown

    display(Markdown(f"## {label}"))
    display(Markdown(f"**Query:** {result.query}"))
    display(Markdown(
        f"**Template:** `{result.template_used}`  \n"
        f"**Cypher:**\n```cypher\n{(result.cypher_query or '(none — no template matched)')[:500]}\n```"
    ))

    if result.graph_hits:
        lines = [f"- Hit {i}: doc={h.doc_id}, page={h.page}, confidence={h.confidence}"
                 for i, h in enumerate(result.graph_hits[:5], 1)]
        display(Markdown("**Graph path / hits:**\n" + "\n".join(lines)))
    else:
        display(Markdown("**Graph path / hits:** *(none — 0 Cypher rows)*"))

    chunk_lines = [f"`{c.chunk_id}` | {c.source_type} | score={c.combined_score:.3f} | p.{c.page}"
                   for c in result.blended_chunks[:4]]
    display(Markdown("**Supporting chunks (top 4):**\n" + "\n".join(f"- {l}" for l in chunk_lines)))

    display(Markdown(
        f"**Retrieval type:** `{result.retrieval_type}`  \n"
        f"**Fallback used:** `{result.retrieval_type in ('hybrid_fallback', 'empty')}`  \n"
        f"**Latency:** {result.latency_sec}s"
    ))
    display(Markdown(f"**Answer:**\n{result.answer}"))
    cit = "; ".join(result.citation_pages) if result.citation_pages else "*(none resolved)*"
    display(Markdown(f"**Citations:** {cit}"))
    display(Markdown(f"**Analysis:** {result.answer_quality_notes}"))
    display(Markdown("---"))

print("display_result() defined.")

display_result() defined.


## 3. Example 1 — Country → Policy → Target
`graphrag_country_policy_target`: `(c:Country)-[:HAS_POLICY]->(p:Policy)-[:SETS_TARGET]->(t:Target)`

Why this works: "UAE" resolves via `ENTITY_ALIASES` to `country_id=ARE`. Vector search would return documents *mentioning* UAE renewables; only the graph can return the exact Policy→Target chain with page anchors.

In [5]:
query_1 = "What makes the FeederBW low-voltage grid dataset useful for energy-transition planning research?"
result_1 = executor.run(query_1)
display_result(result_1, label="Example 1: Energy dataset evidence (FeederBW)")


No GEMINI_API_KEY found; returning prompt as mock answer.


## Example 1: Energy dataset evidence (FeederBW)

**Query:** What makes the FeederBW low-voltage grid dataset useful for energy-transition planning research?

**Template:** `graphrag_finding_document_grounding`  
**Cypher:**
```cypher

        MATCH (f:Finding)-[:SUPPORTED_BY]->(d:Document)
        OPTIONAL MATCH (f)-[:EVIDENCES_RISK]->(risk:ClimateRisk)
        OPTIONAL MATCH (f)-[:EVIDENCES_SECTOR]->(sector:Sector)
        OPTIONAL MATCH (f)-[:EVIDENCES_COUNTRY]->(country:Country)
        OPTIONAL MATCH (f)-[:EVIDENCES_TECHNOLOGY]->(tech:Technology)
        OPTIONAL MATCH (d)-[:PUBLISHED_BY]->(org:Organization)
        WHERE ($risk_id IS NULL OR risk.risk_id = $risk_id)
          AND ($sector_id IS NULL OR sector.sector_id 
```

**Graph path / hits:**
- Hit 1: doc=treutlein_2026_real_world_energy_data_200_feeders_low_voltage_2602_03521v1, page=1, confidence=high
- Hit 2: doc=smith_2019_current_future_role_haber_bosch_ammonia_carbon_free_w2998496658, page=2, confidence=high
- Hit 3: doc=shah_2025_assessing_climate_vulnerability_risk_substations_massachusetts_sensitivity_2511_03748v1, page=3, confidence=high
- Hit 4: doc=alkhayuon_2019_basin_bifurcations_oscillatory_instability_rate_induced_thresholds_amoc_1901_10111v2, page=17, confidence=high
- Hit 5: doc=fatima_2020_fingerprints_climate_warming_cereal_crops_phenology_adaptation_options_w3094545424, page=11, confidence=high

...

**Retrieval type:** `graph_guided`  
**Fallback used:** `False`  
**Latency:** 3.112s

**Answer:**
[MOCK - no API key]

You are a Climate Evidence Research Assistant. You answer questions about climate policy, risks, technologies, and findings using only the provided evidence. You never fabricate citations, page numbers, or statistics. If the evidence does not support a claim, say so explicitly.

## USER QUERY
What makes the FeederBW low-voltage grid dataset useful for energy-transition planning research?

## SECTION 1: GRAPH-GUIDED EVIDENCE (structured, confidence-annotated)
**Knowledge graph path:**
Template: g

**Citations:** (Treutlein et al., 2026, p. 1); (Treutlein et al., 2026, p. 3); (Becker et al., 2024, p. 3); (Becker et al., 2024, p. 4); (Treutlein et al., 2026, p. 19); (Treutlein et al., 2026, p. 18); (Treutlein et al., 2026, p. 6); (Treutlein et al., 2026, p. 2); (Treutlein et al., 2026, p. 4); (Treutlein et al., 2026, p. 17)

**Analysis:** 

---

## 4. Example 2 — Risk → Sector → Evidence
`graphrag_region_climate_risk`: `(region)<-[:LOCATED_IN]-(country)`, `(risk)-[:OCCURS_IN]->(region)`, `(risk)-[:IMPACTS]->(sector)`

Why this works: this is a genuine multi-hop traversal (Region â† Country, Risk → Region, Risk → Sector) that vector search cannot reconstruct — it would retrieve documents mentioning MENA and flooding, but not the structured risk-to-sector impact chain with `Finding`-level evidence pages.

In [6]:
query_2 = "How can sowing-date adjustment help cereal crops respond to warming-driven phenology changes?"
result_2 = executor.run(query_2)
display_result(result_2, label="Example 2: Crop adaptation evidence (sowing-date adjustment)")


No GEMINI_API_KEY found; returning prompt as mock answer.


## Example 2: Crop adaptation evidence (sowing-date adjustment)

**Query:** How can sowing-date adjustment help cereal crops respond to warming-driven phenology changes?

**Template:** `graphrag_finding_document_grounding`  
**Cypher:**
```cypher

        MATCH (f:Finding)-[:SUPPORTED_BY]->(d:Document)
        OPTIONAL MATCH (f)-[:EVIDENCES_RISK]->(risk:ClimateRisk)
        OPTIONAL MATCH (f)-[:EVIDENCES_SECTOR]->(sector:Sector)
        OPTIONAL MATCH (f)-[:EVIDENCES_COUNTRY]->(country:Country)
        OPTIONAL MATCH (f)-[:EVIDENCES_TECHNOLOGY]->(tech:Technology)
        OPTIONAL MATCH (d)-[:PUBLISHED_BY]->(org:Organization)
        WHERE ($risk_id IS NULL OR risk.risk_id = $risk_id)
          AND ($sector_id IS NULL OR sector.sector_id 
```

**Graph path / hits:**
- Hit 1: doc=fatima_2020_fingerprints_climate_warming_cereal_crops_phenology_adaptation_options_w3094545424, page=12, confidence=high
- Hit 2: doc=forster_2020_current_future_global_climate_impacts_resulting_covid_19_w3048190739, page=6, confidence=high
- Hit 3: doc=fatima_2020_fingerprints_climate_warming_cereal_crops_phenology_adaptation_options_w3094545424, page=11, confidence=high
- Hit 4: doc=riahi_2011_rcp_8_5_scenario_comparatively_high_greenhouse_gas_w1997065014, page=1, confidence=high
- Hit 5: doc=scheller_2021_stakeholder_dynamics_residential_solar_energy_adoption_findings_focus_2104_14240v1, page=19, confidence=high

...

**Retrieval type:** `graph_guided`  
**Fallback used:** `False`  
**Latency:** 1.404s

**Answer:**
[MOCK - no API key]

You are a Climate Evidence Research Assistant. You answer questions about climate policy, risks, technologies, and findings using only the provided evidence. You never fabricate citations, page numbers, or statistics. If the evidence does not support a claim, say so explicitly.

## USER QUERY
How can sowing-date adjustment help cereal crops respond to warming-driven phenology changes?

## SECTION 1: GRAPH-GUIDED EVIDENCE (structured, confidence-annotated)
**Knowledge graph path:**
Template: grap

**Citations:** (Fatima et al., 2020, p. 11); (Fatima et al., 2020, p. 6); (Fatima et al., 2020, p. 13); (Fatima et al., 2020, p. 5); (Fatima et al., 2020, p. 7); (Fatima et al., 2020, p. 3); (Fatima et al., 2020, p. 16); (Fatima et al., 2020, p. 15); (Fatima et al., 2020, p. 10); (Fatima et al., 2020, p. 4)

**Analysis:** 

---

## 5. Example 3 — Technology → Mitigates → Risk
`graphrag_technology_mitigates_risk`: `(tech:Technology)-[:MITIGATES]->(risk:ClimateRisk)`

Why this works: `MITIGATES` edges carry a `confidence` and `evidence_page` property. Vector search retrieves documents *about* these technologies but cannot determine that a specific Technology mitigates a specific Risk — that causal claim only exists as a graph edge.

In [7]:
query_3 = "Why do climate projections create vulnerability concerns for electrical substations and transformer loading practices?"
result_3 = executor.run(query_3)
display_result(result_3, label="Example 3: Grid resilience evidence (substations and transformer loading)")


No GEMINI_API_KEY found; returning prompt as mock answer.


## Example 3: Grid resilience evidence (substations and transformer loading)

**Query:** Why do climate projections create vulnerability concerns for electrical substations and transformer loading practices?

**Template:** `graphrag_finding_document_grounding`  
**Cypher:**
```cypher

        MATCH (f:Finding)-[:SUPPORTED_BY]->(d:Document)
        OPTIONAL MATCH (f)-[:EVIDENCES_RISK]->(risk:ClimateRisk)
        OPTIONAL MATCH (f)-[:EVIDENCES_SECTOR]->(sector:Sector)
        OPTIONAL MATCH (f)-[:EVIDENCES_COUNTRY]->(country:Country)
        OPTIONAL MATCH (f)-[:EVIDENCES_TECHNOLOGY]->(tech:Technology)
        OPTIONAL MATCH (d)-[:PUBLISHED_BY]->(org:Organization)
        WHERE ($risk_id IS NULL OR risk.risk_id = $risk_id)
          AND ($sector_id IS NULL OR sector.sector_id 
```

**Graph path / hits:**
- Hit 1: doc=chu_2016_path_sustainable_energy_w2564178799, page=6, confidence=high
- Hit 2: doc=bui_2018_carbon_capture_storage_ccs_way_forward_w2784350055, page=84, confidence=high
- Hit 3: doc=smith_2019_current_future_role_haber_bosch_ammonia_carbon_free_w2998496658, page=2, confidence=high
- Hit 4: doc=shah_2025_assessing_climate_vulnerability_risk_substations_massachusetts_sensitivity_2511_03748v1, page=3, confidence=high
- Hit 5: doc=riahi_2011_rcp_8_5_scenario_comparatively_high_greenhouse_gas_w1997065014, page=1, confidence=high

...

**Retrieval type:** `graph_guided`  
**Fallback used:** `False`  
**Latency:** 2.549s

**Answer:**
[MOCK - no API key]

You are a Climate Evidence Research Assistant. You answer questions about climate policy, risks, technologies, and findings using only the provided evidence. You never fabricate citations, page numbers, or statistics. If the evidence does not support a claim, say so explicitly.

## USER QUERY
Why do climate projections create vulnerability concerns for electrical substations and transformer loading practices?

## SECTION 1: GRAPH-GUIDED EVIDENCE (structured, confidence-annotated)
**Knowledge gra

**Citations:** (Rosenzweig et al., 2014, p. 10); (Shah et al., 2025, p. 1); (Shah et al., 2025, p. 3); (Cardenas, 2024, p. 2); (Chu et al., 2016, p. 6); (Shah et al., 2025, p. 2); (Shah et al., 2025, p. 6); (Shah et al., 2025, p. 4); (Rosenzweig et al., 2014, p. 5); (Rosenzweig et al., 2014, p. 7); (Rosenzweig et al., 2014, p. 2)

**Analysis:** 

---

## 6. Required failure case — Cypher expansion too broad
`graphrag_finding_document_grounding` with `risk_id=None`: with no entity filter, this returns **every** `Finding` node ordered by confidence. The top-ranked findings cluster around the same 2–3 IPCC AR6 pages, so after expansion the supporting chunks are near-duplicates instead of diverse evidence.

**What vector search would do better here:** semantically diverse chunks across multiple documents, giving broader topical coverage of the same statistic.

In [8]:
query_4 = "How could carbon-free Haber-Bosch ammonia production align with intermittent renewable energy?"
result_4 = executor.run(query_4)
display_result(result_4, label="Example 4: Technology evidence (carbon-free ammonia)")


No GEMINI_API_KEY found; returning prompt as mock answer.


## Example 4: Technology evidence (carbon-free ammonia)

**Query:** How could carbon-free Haber-Bosch ammonia production align with intermittent renewable energy?

**Template:** `graphrag_finding_document_grounding`  
**Cypher:**
```cypher

        MATCH (f:Finding)-[:SUPPORTED_BY]->(d:Document)
        OPTIONAL MATCH (f)-[:EVIDENCES_RISK]->(risk:ClimateRisk)
        OPTIONAL MATCH (f)-[:EVIDENCES_SECTOR]->(sector:Sector)
        OPTIONAL MATCH (f)-[:EVIDENCES_COUNTRY]->(country:Country)
        OPTIONAL MATCH (f)-[:EVIDENCES_TECHNOLOGY]->(tech:Technology)
        OPTIONAL MATCH (d)-[:PUBLISHED_BY]->(org:Organization)
        WHERE ($risk_id IS NULL OR risk.risk_id = $risk_id)
          AND ($sector_id IS NULL OR sector.sector_id 
```

**Graph path / hits:**
- Hit 1: doc=smith_2019_current_future_role_haber_bosch_ammonia_carbon_free_w2998496658, page=2, confidence=high
- Hit 2: doc=scheller_2021_stakeholder_dynamics_residential_solar_energy_adoption_findings_focus_2104_14240v1, page=19, confidence=high
- Hit 3: doc=ravi_2025_citizen_centered_climate_intelligence_operationalizing_open_tree_data_2508_17648v1, page=7, confidence=high
- Hit 4: doc=treutlein_2026_real_world_energy_data_200_feeders_low_voltage_2602_03521v1, page=1, confidence=high
- Hit 5: doc=riahi_2011_rcp_8_5_scenario_comparatively_high_greenhouse_gas_w1997065014, page=1, confidence=high

...

**Retrieval type:** `graph_guided`  
**Fallback used:** `False`  
**Latency:** 1.418s

**Answer:**
[MOCK - no API key]

You are a Climate Evidence Research Assistant. You answer questions about climate policy, risks, technologies, and findings using only the provided evidence. You never fabricate citations, page numbers, or statistics. If the evidence does not support a claim, say so explicitly.

## USER QUERY
How could carbon-free Haber-Bosch ammonia production align with intermittent renewable energy?

## SECTION 1: GRAPH-GUIDED EVIDENCE
*(No graph paths found for this query; answer based on Section 2 only.)*



**Citations:** (Smith et al., 2019, p. 1); (Smith et al., 2019, p. 11); (Smith et al., 2019, p. 7); (Smith et al., 2019, p. 9); (Smith et al., 2019, p. 10); (Smith et al., 2019, p. 6); (Smith et al., 2019, p. 4); (Smith et al., 2019, p. 2); (Smith et al., 2019, p. 8)

**Analysis:** 

---

## 7. Additional failure cases (for completeness)
Two more documented failure modes, run here so the canonical CSV has the full set of 3 success + 3 failure rows.

In [9]:
query_5 = "How could post-COVID green stimulus choices affect future warming trajectories?"
result_5 = executor.run(query_5)
display_result(result_5, label="Example 5: Mitigation evidence (post-COVID green stimulus)")


No GEMINI_API_KEY found; returning prompt as mock answer.


## Example 5: Mitigation evidence (post-COVID green stimulus)

**Query:** How could post-COVID green stimulus choices affect future warming trajectories?

**Template:** `graphrag_finding_document_grounding`  
**Cypher:**
```cypher

        MATCH (f:Finding)-[:SUPPORTED_BY]->(d:Document)
        OPTIONAL MATCH (f)-[:EVIDENCES_RISK]->(risk:ClimateRisk)
        OPTIONAL MATCH (f)-[:EVIDENCES_SECTOR]->(sector:Sector)
        OPTIONAL MATCH (f)-[:EVIDENCES_COUNTRY]->(country:Country)
        OPTIONAL MATCH (f)-[:EVIDENCES_TECHNOLOGY]->(tech:Technology)
        OPTIONAL MATCH (d)-[:PUBLISHED_BY]->(org:Organization)
        WHERE ($risk_id IS NULL OR risk.risk_id = $risk_id)
          AND ($sector_id IS NULL OR sector.sector_id 
```

**Graph path / hits:**
- Hit 1: doc=forster_2020_current_future_global_climate_impacts_resulting_covid_19_w3048190739, page=6, confidence=high
- Hit 2: doc=scheller_2021_stakeholder_dynamics_residential_solar_energy_adoption_findings_focus_2104_14240v1, page=19, confidence=high
- Hit 3: doc=fatima_2020_fingerprints_climate_warming_cereal_crops_phenology_adaptation_options_w3094545424, page=11, confidence=high
- Hit 4: doc=shah_2025_assessing_climate_vulnerability_risk_substations_massachusetts_sensitivity_2511_03748v1, page=3, confidence=high
- Hit 5: doc=fatima_2020_fingerprints_climate_warming_cereal_crops_phenology_adaptation_options_w3094545424, page=12, confidence=high

...

**Retrieval type:** `graph_guided`  
**Fallback used:** `False`  
**Latency:** 2.304s

**Answer:**
[MOCK - no API key]

You are a Climate Evidence Research Assistant. You answer questions about climate policy, risks, technologies, and findings using only the provided evidence. You never fabricate citations, page numbers, or statistics. If the evidence does not support a claim, say so explicitly.

## USER QUERY
How could post-COVID green stimulus choices affect future warming trajectories?

## SECTION 1: GRAPH-GUIDED EVIDENCE (structured, confidence-annotated)
**Knowledge graph path:**
Template: graphrag_finding_d

**Citations:** (Forster et al., 2020, p. 6); (Zhuo et al., 2022, p. 11); (Becker et al., 2024, p. 41); (Calvin et al., 2023, p. 18); (Forster et al., 2020, p. 1); (Forster et al., 2020, p. 9); (Forster et al., 2020, p. 3)

**Analysis:** 

---

### Failure-case example

The next cell intentionally tests a query that should fall back safely when the graph schema does not directly cover the requested entity or method.


In [10]:
# Fallback: graph adds no value ? "gradient boosting" is an ML method, not a Climate-KG Technology node.
# This is kept as the one deliberate failure/fallback demo.
query_6 = "What does the literature say about gradient boosting methods for wind power forecasting?"
result_6 = executor.run(query_6)
result_6.answer_quality_notes = (
    result_6.answer_quality_notes or
    "FALLBACK CASE: Graph adds no value. 'Gradient boosting' has no Technology node in the Climate-KG; "
    "Cypher returns 0 rows; answer is grounded in hybrid retrieval (retrieval_type=hybrid_fallback). "
    "This demonstrates safe degradation instead of forcing an irrelevant graph path."
)
display_result(result_6, label="Fallback case: Graph adds no value (gradient boosting)")


No GEMINI_API_KEY found; returning prompt as mock answer.


## Fallback case: Graph adds no value (gradient boosting)

**Query:** What does the literature say about gradient boosting methods for wind power forecasting?

**Template:** `graphrag_technology_mitigates_risk`  
**Cypher:**
```cypher

        MATCH (tech:Technology)
        WHERE $tech_id IS NULL OR tech.tech_id = $tech_id
        OPTIONAL MATCH (tech)-[m:MITIGATES]->(risk:ClimateRisk)
        OPTIONAL MATCH (risk)-[:IMPACTS]->(sector:Sector)
        OPTIONAL MATCH (risk)-[:OCCURS_IN]->(region:Region)
        OPTIONAL MATCH (d:Document {doc_id: m.doc_id})
        RETURN tech.name AS technology,
               risk.name AS mitigated_risk,
               m.confidence AS mitigation_confidence,
               sector.name AS appl
```

**Graph path / hits:** *(none — 0 Cypher rows)*

...

**Retrieval type:** `hybrid_fallback`  
**Fallback used:** `True`  
**Latency:** 1.97s

**Answer:**
[MOCK - no API key]

You are a Climate Evidence Research Assistant. You answer questions about climate policy, risks, technologies, and findings using only the provided evidence. You never fabricate citations, page numbers, or statistics. If the evidence does not support a claim, say so explicitly.

## USER QUERY
What does the literature say about gradient boosting methods for wind power forecasting?

## SECTION 1: GRAPH-GUIDED EVIDENCE
*(No graph paths found for this query; answer based on Section 2 only.)*

## SEC

**Citations:** (Pinson, 2013, p. 2); (Ye et al., 2024, p. 2); (Pinson, 2013, p. 4); (Ju et al., 2019, p. 1); (Abuella et al., 2017, p. 1); (Pinson, 2013, p. 21); (Pinson, 2013, p. 23); (Pinson, 2013, p. 6); (Pinson, 2013, p. 3); (Ju et al., 2019, p. 9); (Ye et al., 2024, p. 4); (Pinson, 2013, p. 11); (Pinson, 2013, p. 12); (Pinson, 2013, p. 15)

**Analysis:** FALLBACK CASE: Graph adds no value. 'Gradient boosting' has no Technology node in the Climate-KG; Cypher returns 0 rows; answer is grounded in hybrid retrieval (retrieval_type=hybrid_fallback). This demonstrates safe degradation instead of forcing an irrelevant graph path.

---

## 8. Save canonical results table
Writes `reports/tables/d3_graph_guided_results.csv` via `CitationBuilder.build_csv_row()`, which now includes explicit `fallback_used` and `failure_note` columns (not just the free-text `answer_quality_notes`) plus `latency_sec`.

In [11]:
all_results = [result_1, result_2, result_3, result_4, result_5, result_6]
rows = [citation_builder.build_csv_row(r) for r in all_results]
df = pd.DataFrame(rows)
df["answer_mode"] = "gemini" if USE_GEMINI_ANSWERS else "mock_prompt_no_gemini"
df["graph_node_count"] = GRAPH_NODE_COUNT
df["graph_finding_count"] = GRAPH_FINDING_COUNT

out_path = REPORTS_TABLES / "d3_graph_guided_results.csv"
df.to_csv(out_path, index=False)
print(f"Saved {len(df)} rows to {out_path}")
df[["query", "retrieval_type", "fallback_used", "latency_sec", "answer_mode", "graph_node_count", "graph_finding_count"]]


Saved 6 rows to D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables\d3_graph_guided_results.csv


,query,retrieval_type,fallback_used,latency_sec,answer_mode,graph_node_count,graph_finding_count
0,What makes the FeederBW low-voltage grid datas...,graph_guided,False,3.112,mock_prompt_no_gemini,552,15
1,How can sowing-date adjustment help cereal cro...,graph_guided,False,1.404,mock_prompt_no_gemini,552,15
2,Why do climate projections create vulnerabilit...,graph_guided,False,2.549,mock_prompt_no_gemini,552,15
3,How could carbon-free Haber-Bosch ammonia prod...,graph_guided,False,1.418,mock_prompt_no_gemini,552,15
4,How could post-COVID green stimulus choices af...,graph_guided,False,2.304,mock_prompt_no_gemini,552,15
5,What does the literature say about gradient bo...,hybrid_fallback,True,1.970,mock_prompt_no_gemini,552,15


## 9. Critical design review (deep-evaluation questions)

**Is the graph path actually used in the answer, or just displayed as decoration?**  
Actually used. `PromptBuilder.build_prompt()` writes the graph path summary and graph-sourced chunks into **Section 1** of the LLM prompt and explicitly instructs the model to *"prioritise the GRAPH-GUIDED EVIDENCE in Section 1."* The trace shown above (Cypher → graph hits → chunks) is what actually gets sent to the model, not a separate display-only artifact.

**Are citations built from supporting chunks, or from graph node names alone?**  
From chunks. `CitationBuilder.build()` / `build_from_hits()` resolve citations from `chunk.doc_id` + `chunk.page` (looked up against `papers_metadata.csv` / the chunk store), never from a `Policy.name` or `Technology.name` string directly. A graph node can appear in the *path summary* without ever generating a citation if it isn't backed by a chunk/Finding with a resolvable `doc_id`/`page`.

**Is the executor honest when graph evidence is weak?**  
Yes, via `fallback_used` (boolean, true whenever `retrieval_type` is `hybrid_fallback`/`empty`) and `failure_note` (the failure rationale, populated whenever fallback occurred *or* the notes are explicitly flagged `FAILURE`, e.g. the "too broad" case above where the graph technically returned rows but they were low-value). See the CSV.

**When does the graph genuinely help?**
1. **Country → Policy → Target** — multi-hop chains with no text-proximity equivalent (Example 1).
2. **Risk → Sector → Evidence** — confidence- and page-annotated impact chains (Example 2).
3. **Technology → Mitigates → Risk** — causal claims that only exist as graph edges, not as co-occurring text (Example 3).

**When it doesn't:** no extractable entity, entity missing from the alias table/corpus, over-broad unfiltered expansion, or the question is methodological rather than entity-based (Examples 4–6).

## Limitations and interpretation

1. **Alias table coverage** ? `ENTITY_ALIASES` covers the main D3 demo entities, but entities outside it (for example, Pacific Islands or specific ML algorithms) may fail entity extraction and correctly fall back to hybrid retrieval.
2. **Template coverage** ? the 5 `GRAPHRAG_REASONING` templates are intentionally controlled. They demonstrate country?policy?target, risk?sector?evidence, technology?risk, finding?document grounding, and fallback behavior. More composite questions would need a richer Cypher planner.
3. **Finding sparsity** ? the graph contains a curated D3 set, so sparse topics may return fewer graph hits than broad retrieval.
4. **Gemini is disabled in this notebook on purpose** ? the notebook tests the GraphRAG executor, graph routing, expansion, hybrid blending, citation construction, fallback logic, and latency without consuming Gemini quota. The saved `answer_mode` column is therefore `mock_prompt_no_gemini`.
5. **Latency values are measured for the local no-Gemini pipeline** ? they are useful for comparing retrieval/executor behavior, but not for estimating full LLM answer-generation latency.

## Before-submission checklist
- [x] Run this notebook top-to-bottom against the configured Neo4j graph from `.env`.
- [x] Keep Gemini disabled for stable reproducible submission reruns.
- [x] Confirm the graph contains the expected D3 scale: 552 nodes / 15 findings.
- [x] Confirm the CSV has 6 rows with `fallback_used`, `failure_note`, `latency_sec`, `answer_mode`, `graph_node_count`, and `graph_finding_count`.
- [x] Keep one or more fallback cases because that demonstrates safe degradation when the graph cannot answer a query directly.
- [ ] If the instructor asks for live answer-generation screenshots, temporarily set `USE_GEMINI_ANSWERS=True` and rerun only the example cells, not the whole evaluation repeatedly.
